# **IMPORTS**

In [ ]:
# Apache Spark API
from pyspark.sql import SparkSession

# Apache Spark Types
from pyspark.sql.types import *

# Apache Spark SQL utils
from pyspark.sql.functions import *

# Apache Spark Machine Learning
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import NaiveBayes, MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Data Handling
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt

In [ ]:
# Iniciando Spark Session
spark = SparkSession.builder \
    .appName("BigData_AC2") \
    .master("local[*]") \
    .config("spark.executor.memory", "12g") \
    .config("spark.driver.memory",   "12g") \
    .config("spark.cleaner.referenceTracking.blocking", "true") \
    .config("spark.cleaner.referenceTracking.cleanCheckpoints", "true") \
    .config("spark.storage.cleanupFilesAfterExecutorExit", "true") \
    .getOrCreate()

In [ ]:
# Definição de Schema dos dados
df = spark.read.schema(
    StructType([
      StructField(name="Year", dataType=LongType(), nullable=True),
      StructField(name="Month", dataType=LongType(), nullable=True),
      StructField(name="DayofMonth", dataType=LongType(), nullable=True),
      StructField(name="DayOfWeek", dataType=LongType(), nullable=True),
      StructField(name="DepTime", dataType=DoubleType(), nullable=True),
      StructField(name="CRSDepTime", dataType=LongType(), nullable=True),
      StructField(name="ArrTime", dataType=DoubleType(), nullable=True),
      StructField(name="CRSArrTime", dataType=LongType(), nullable=True),
      StructField(name="UniqueCarrier", dataType=StringType(), nullable=True),
      StructField(name="FlightNum", dataType=LongType(), nullable=True),
      StructField(name="TailNum", dataType=StringType(), nullable=True),
      StructField(name="ActualElapsedTime", dataType=DoubleType(), nullable=True),
      StructField(name="CRSElapsedTime", dataType=DoubleType(), nullable=True),
      StructField(name="AirTime", dataType=DoubleType(), nullable=True),
      StructField(name="ArrDelay", dataType=DoubleType(), nullable=True),
      StructField(name="DepDelay", dataType=DoubleType(), nullable=True),
      StructField(name="Origin", dataType=StringType(), nullable=True),
      StructField(name="Dest", dataType=StringType(), nullable=True),
      StructField(name="Distance", dataType=DoubleType(), nullable=True),
      StructField(name="TaxiIn", dataType=DoubleType(), nullable=True),
      StructField(name="TaxiOut", dataType=DoubleType(), nullable=True),
      StructField(name="Cancelled", dataType=LongType(), nullable=True),
      StructField(name="CancellationCode", dataType=StringType(), nullable=True),
      StructField(name="CarrierDelay", dataType=DoubleType(), nullable=True),
      StructField(name="WeatherDelay", dataType=DoubleType(), nullable=True),
      StructField(name="NASDelay", dataType=DoubleType(), nullable=True),
      StructField(name="SecurityDelay", dataType=DoubleType(), nullable=True),
      StructField(name="LateAircraftDelay", dataType=DoubleType(), nullable=True)
    ])
).parquet("../../../data/airline.parquet", header=True)
print(f"N° samples: {df.count()}")

# **DATA CLEANING**

### MISSING DATA ANALYSIS BEFORE

In [ ]:
# Criando tabela para análise de valores faltantes
missing = df.select([
    sum(
        when( col(c).isNull() | isnan(col(c)), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
])

In [ ]:
# Aplicando transposição na tebela para orientar as colunas para linhas, e melhorar a visualização
missing = missing.selectExpr(
        "stack({0}, {1}) as (column, missing)".format(
            len(df.columns),
            ", ".join("'{}', {}".format(c, c) for c in df.columns)
        )
    ).withColumn(
        "porcentage",
        round(col("missing") / df.count() * 100, 2)
    ).orderBy(desc("missing"))

missing.show(truncate=False)

In [ ]:
# Removendo colunas com muitos dados faltantes
df = df.drop(*['LateAircraftDelay', 'SecurityDelay', 'NASDelay', 'WeatherDelay', 'CarrierDelay', 'CancellationCode'])

# Removendo linhas com dados faltantes da coluna 'TailNum'
# Essa coluna representa o código da asa do avião, ou seja, não é possível preencher usando alguma técnica estatísca
df = df.na.drop(how='any', subset=('TailNum'))

### BALANCING ANALYSIS

In [ ]:
# Dados desbalanceados na coluna alvo original, sendo muito difícil tratar esse desbalanceamento
(df.groupBy('Cancelled').agg(
    count("Cancelled").alias("count"),
    (count("Cancelled") / df.count()).alias("percentage")
).show())

In [ ]:
# N° de linhas com NULL values da coluna ArrDelay
display(df.select('ArrDelay').where(col('ArrDelay').isNull()).count())

# N° de linhas com NaN values da coluna ArrDelay
display(df.select('ArrDelay').where(isnan('ArrDelay')).count())

In [ ]:
# Remoção de valores NULL da coluna ArrDelay
# Justamente para criar manter apenas linhas com informação e criar a nova coluna
df = df.filter(col("ArrDelay").isNotNull())

In [ ]:
# Criando nova coluna alvo para análise de desbalanceamento
# Coluna que representa se o voo atrasou ou não em sua chegada
df = df.withColumn(
    "IsDelay",
    when(col("ArrDelay") > 0, True).otherwise(False).cast(IntegerType())
)

In [ ]:
# Visualizando os impactos no balanceamento da coluna alvo
(df.groupBy('IsDelay').agg(
    count("IsDelay").alias("count"),
    (count("IsDelay") / df.count()).alias("percentage")
).show())

### OUTLIER ANALYSIS

#### 'ArrDelay' COLUMN TURKEY ANALYSIS

Tukey's method:

IQR = $Q3 - Q1$

lower threshold = $Q1 − 1.5 × IQR$

upper threshold = $Q3 + 1.5 × IQR$

In [ ]:
# Cálculo de quartis usando Spark
quantiles = df.approxQuantile('ArrDelay', [0.25, 0.75], 0.0001)

# Calcula threshold aplicando o método de Turkey
#             Q1 - 1.1 * (Q3 - Q1)
lower_bound = quantiles[0] - 1.1 * (quantiles[1] - quantiles[0])
#             Q3 + 1.1 * (Q3 - Q1)
upper_bound = quantiles[1] + 1.1 * (quantiles[1] - quantiles[0])

# Neste caso, não queremos uma coluna perfeita, sem outliers, ou perder informação importante
# Assim, usamos uma constante de 1.2 no cálculo

In [ ]:
# Número de outliers na coluna 'ArrDelay'
df.select('ArrDelay').filter((col('ArrDelay') < lower_bound) | (col('ArrDelay') > upper_bound)).count()

In [ ]:
# Remoção de valores anômalos da análise Turkey da coluna ArrDelay
df = df.filter((col('ArrDelay') >= lower_bound) & (col('ArrDelay') <= upper_bound))

In [ ]:
# Após remoção dos outliers, o desbalanceamento aumentou na coluna alvo
(df.groupBy('IsDelay').agg(
    count("IsDelay").alias("count"),
    (count("IsDelay") / df.count()).alias("percentage")
).show())

### MISSING DATA ANALYSIS AFTER

In [ ]:
# Criando tabela para análise de valores faltantes
missing = df.select([
    sum(
        when( col(c).isNull() | isnan(col(c)), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
])

In [ ]:
# Aplicando transposição na tebela para orientar as colunas para linhas, e melhorar a visualização
missing = missing.selectExpr(
    "stack({0}, {1}) as (column, missing)".format(
        len(df.columns),
        ", ".join("'{}', {}".format(c, c) for c in df.columns)
    )
).withColumn(
    "porcentage",
    round(col("missing") / df.count() * 100, 4)
).orderBy(desc("missing"))

missing.show(truncate=False)

In [ ]:
# Visualizando distribuição da coluna Distance
# Distribuição conhecida como Power Law, que por característica, não possui média em seu significado
edges, counts = (
    df.select("Distance").rdd.flatMap(
        lambda row: row
    ).histogram(18) # qunatidade de barras no plot | aumentando o número de barras para analisar com mais precisão a distribuição
)

plt.figure(figsize=(8,5))
plt.bar(edges[:-1], counts, width=np.diff(edges), align="edge")
plt.xlabel('data')
plt.ylabel('frequency')
plt.title('Distance Distribution')
plt.show()

In [ ]:
# Preenchendo com mediana as linhas onde o valor da coluna Distance é nulo
# Mediana é uma medida de localidade, menos sensível a outliers
df = df.na.fill({'Distance': df.approxQuantile("Distance", [0.5], 0.01)[0]})

In [ ]:
# Visualizando distribuição da coluna AirTime
edges, counts = (
    df.select("AirTime").rdd.flatMap(
        lambda row: row
    ).histogram(8) # qunatidade de barras no plot
)

plt.figure(figsize=(8,5))
plt.bar(edges[:-1], counts, width=np.diff(edges), align="edge")
plt.xlabel('data')
plt.ylabel('frequency')
plt.title('AirTime Distribution')
plt.show()

In [ ]:
# Preenchido valores faltantes com a mediana da coluna
# Mediana é uma medida de localidade, menos sensível a outliers
df = df.na.fill({'AirTime': df.approxQuantile("AirTime", [0.5], 0.01)[0]})

### DISTRIBUTION ANALYSIS

In [ ]:
# Após a limpeza realizada anteriormente, percebe-se que os dados da coluna alvo tendem para uma distribuição normal
# Um bom sinal, representando uma coluna com potêncial e qualidade na informação
edges, counts = (
    df.select("ArrDelay").rdd.flatMap(
        lambda row: row
    ).histogram(8) # qunatidade de barras no plot
)

plt.figure(figsize=(8,5))
plt.bar(edges[:-1], counts, width=np.diff(edges), align="edge")
plt.xlabel('data')
plt.ylabel('frequency')
plt.title('ArrDely Distribution')
plt.show()

## **PERSIST CLEANED DATA**

In [ ]:
try:
    raw.repartition(1).write.mode('overwrite').parquet('../../../data/airline_clean.parquet')
except Exception as e:
    print(e)

# **EXPLORATION DATA ANALYSIS**

Descrição sumarizada do Kaggle:

Os dados consistem em detalhes de chegada e partida de todos os voos comerciais nos EUA, de outubro de 1987 a abril de 2008. Trata-se de um grande conjunto de dados: há quase 120 milhões de registros no total.

Você também pode trabalhar com subconjuntos interessantes: talvez queira comparar os padrões de voo antes e depois do 11 de setembro, ou entre os pares de cidades entre as quais você voa com mais frequência, ou todos os voos de e para um aeroporto importante como Chicago (ORD).

### DATA TREATMENT

In [ ]:
# Normalizando colunas do tipo string
df = df.withColumn("Origin", upper(col('Origin')))

### QUESTÃO 1 - Qual é a melhor dia da semana/época do ano para voar e minimizar os atrasos?

In [ ]:
# Melhor mês do ano para viajar, com maiores chances de não atrasar e não cancelar cancelar é em setembro
(
    df.groupBy(
        'Month', 'isDelay', 'Cancelled'
    ).agg(
        round((count("isDelay") / df.count())*100, 4).alias("probability_delay"),
        round((count("Cancelled") / df.count())*100, 4).alias("probability_cancelled")
    ).filter(
        (col('isDelay') == 0) & (col('Cancelled') == 0)
    ).orderBy(
        *['probability_delay', 'probability_cancelled'], ascending=False
).show())

In [ ]:
# O melhor dia da semana, que tem maior probabilidade de não atrasar e não cancelar é na terça-feira
(
    df.groupBy(
        'DayOfWeek', 'isDelay', 'Cancelled'
    ).agg(
        round((count("isDelay") / df.count())*100, 4).alias("probability_delay"),
        round((count("Cancelled") / df.count())*100, 4).alias("probability_cancelled")
    ).filter(
        (col('isDelay') == 0) & (col('Cancelled') == 0)
    ).orderBy(
        *['probability_delay', 'probability_cancelled'], ascending=False
).show())

In [ ]:
# Os melhores dias do mês, para ter maior chance de não atrasar e não cancelar o voo, é nos primeiros dias do mês.
(
    df.groupBy(
        'DayofMonth', 'isDelay', 'Cancelled'
    ).agg(
        round((count("isDelay") / df.count())*100, 4).alias("probability_delay"),
        round((count("Cancelled") / df.count())*100, 4).alias("probability_cancelled")
    ).filter(
        col('isDelay') == 0
    ).orderBy(
        *['probability_delay', 'probability_cancelled'], ascending=False
).show())

### QUESTÃO 2 - Como o número de pessoas que voam entre diferentes locais muda ao longo do tempo?

In [ ]:
# 1 https://pt.wikipedia.org/wiki/Atlanta -> Referência de justificativa do aumento de voos para atlanta na década de 2000
# 2 Em 2000, o Eppley Airfield de Omaha era o maior aeroporto de Nebraska, atendendo a 5 milhões de passageiros por ano.
# Ele oferecia serviço sem parada para 31 dos aeroportos mais movimentados do país e tinha mais de 200 chegadas e partidas diárias.
# Embora não seja específico para voos entre o ORD (Aeroporto Internacional O'Hare de Chicago) e Nebraska, essas informações indicam
# que Omaha era um importante centro de viagens aéreas em Nebraska.
# 3 https://en.wikipedia.org/wiki/Dallas_Fort_Worth_International_Airport -> Um dos maiores aeroportos dos Estados Unidos

(
    df.groupBy(
        'Year','Dest'
    ).agg(
         count("Dest").alias("count"),
    ).orderBy(
        *['count', 'Year'], ascending=False
).show())

# 1 Atlanta
# 2 ORD Nebraska
# 3 DFW (Dallas Fort Worth International Airport)

In [ ]:
# Percebe-se o aumento de viagens ao longo do tempo para o aeroporto de Atlanta
# Ainda, a melhoria na infraestrutura do aeroporto, diminuindo o atraso de voos ao longo do tempo
# Um grande aumento nos voos com destino para Atlanta, na década dos anos 2000
plot_data = df.filter(
    (col('Dest') == 'ATL')
).groupBy(
    'Dest', 'IsDelay', 'Year'
).agg(
    count("IsDelay").alias("count")
).orderBy('Year')

x = plot_data.filter(col('isDelay') == 1).select('Year').rdd.flatMap(list).collect()
y1 = plot_data.filter(col('isDelay') == 1).select('count').rdd.flatMap(list).collect()
y2 = plot_data.filter(col('isDelay') == 0).select('count').rdd.flatMap(list).collect()

plt.figure(figsize=(9,3))
plt.plot(x, y1, label='Delay', marker='o', linestyle='dashed')
plt.plot(x, y2, label='No Delay', marker='o', linestyle='dashed')
plt.xlabel('year')
plt.title('Delay along the time from Atlanta airport')
plt.legend()
plt.show()

In [ ]:
# Aumento nos voos para o aeroporto de ORD, devido por ser muito utilizado nas rotas dos voos
plot_data = df.filter(
    (col('Dest') == 'ORD')
).groupBy(
    'Dest', 'IsDelay', 'Year'
).agg(
    count("IsDelay").alias("count")
).orderBy('Year')

x = plot_data.filter(col('isDelay') == 1).select('Year').rdd.flatMap(list).collect()
y1 = plot_data.filter(col('isDelay') == 1).select('count').rdd.flatMap(list).collect()
y2 = plot_data.filter(col('isDelay') == 0).select('count').rdd.flatMap(list).collect()

plt.figure(figsize=(9,3))
plt.plot(x, y1, label='Delay', marker='o', linestyle='dashed')
plt.plot(x, y2, label='No Delay', marker='o', linestyle='dashed')
plt.xlabel('year')
plt.title('Delay along the time from ORD airport')
plt.legend()
plt.show()

### QUESTÃO 3 - Comparação dos padrões de voo antes e depois do 11 de setembro de 2001

In [ ]:
# Percebe-se uma queda nos voos após o ano de 2001, e uma grande volta no número de voos após 2002, nos Estados Unidos
plot_data = df.groupBy('Year').agg(
     count("Year").alias("count")
).orderBy('Year')

x = plot_data.select('Year').rdd.flatMap(list).collect()
y1 = plot_data.select('count').rdd.flatMap(list).collect()

plt.figure(figsize=(9,3))
plt.plot(x, y1, marker='o', linestyle='dashed')
plt.xlabel('year')
plt.title('Number of flights along the time')
plt.show()

# **PRE-PROCESSING DATA**

### DATA CLEANING

In [ ]:
# Removendo colunas que não são viáveis aplicar OneHotEnconding
# São colunas com alto número de valores categóricos
# Aplicar OneHotEncoding nessas colunas, vai deixar o dataset esparso
df = df.drop(*['Dest', 'Origin', 'TailNum', 'FlightNum', 'Cancelled', 'DayofMonth', 'UniqueCarrier', 'DayOfWeek', 'Year'])

### SAMPLING DATA

In [ ]:
# Random sampling, separando 80% do dataset para treino e 20% para teste
train, test = df.randomSplit([0.8, 0.2], seed=2025)

display(train.count())
display(test.count())

### BALANCING TARGET COLUMN

In [ ]:
# Visualizando a proporção do desbalanceamento entre as classes da coluna alvo
(train.groupBy('IsDelay').agg(
    count("IsDelay").alias("count"),
    (count("IsDelay") / train.count()).alias("percentage")
).show())

In [ ]:
# Cálculo usando Regra de 3, para achar a porcentagem para aplicar o undersampling
(100*12332089)/35768669 - 100

In [ ]:
# Aplicando undersampling na classe majoritária
train = train.filter(
    col('isDelay') == 0
).sample(
    withReplacement=False, fraction=0.655, seed=2025
).unionByName(train.filter(col('isDelay') == 1))

In [ ]:
# Verificando o balanceamento
(train.groupBy('IsDelay').agg(
    count("IsDelay").alias("count"),
    (count("IsDelay") / train.count()).alias("percentage")
).show())

### NORMALIZATION WITH Z-SCORE

$Z_{score} = \frac{x_i - μ}{σ}$

In [ ]:
# Definindo colunas numéricas
columns_to_normalize = [
 'DepTime',
 'CRSDepTime',
 'ArrTime',
 'CRSArrTime',
 'ActualElapsedTime',
 'CRSElapsedTime',
 'AirTime',
 'ArrDelay',
 'DepDelay',
 'Distance',
 'TaxiIn',
 'TaxiOut'
]

# Calculando média e desvio padrão do set de treino, para cálculo do Z-Score
# Necessário usar média e desvio padrão do set de treino, para normalizar o set de teste
stats = (
    train.agg(
        *([mean(column).alias(f"{column}_mean") for column in columns_to_normalize] +
         [std(column).alias(f"{column}_std")  for column in columns_to_normalize])
    ).first().asDict()
)

In [ ]:
# Aplicando fórmula do Z-score no set de treino
for column in columns_to_normalize:
    train = train.withColumn(
        column,
        (col(column) - stats[f"{column}_mean"]) / stats[f"{column}_std"]
    )

# Aplicando fórmula do Z-score no set de teste
for column in columns_to_normalize:
    test = test.withColumn(
        column,
        (col(column) - stats[f"{column}_mean"]) / stats[f"{column}_std"]
    )

### ONE HOT ENCONDING

In [ ]:
# Transformando coluna para tipo string
train = train.withColumn("Month", col("Month").cast(StringType()))
# Adicionando prefixo para colunas do one hot encoding
train = train.withColumn("Month", concat(lit('Month_'), col("Month")))

# Transformando coluna para tipo string
test = test.withColumn("Month", col("Month").cast(StringType()))
# Adicionando prefixo para colunas do one hot encoding
test = test.withColumn("Month", concat(lit('Month_'), col("Month")))

In [ ]:
# One hot encoding coluna Month do treino
train = train.groupBy(
    train.drop('Month').columns
).pivot('Month').agg(
    count('*')
).fillna(0)

# One hot encoding coluna Month do teste
test = test.groupBy(
    test.drop('Month').columns
).pivot('Month').agg(
    count('*')
).fillna(0)

## **PERSIST PROCESSED DATA**

In [ ]:
try:
    train.repartition(1).write.mode('overwrite').parquet('../../../data/train.parquet')
except Exception as e:
    print(e)

In [ ]:
try:
    test.repartition(1).write.mode('overwrite').parquet('../../../data/test.parquet')
except Exception as e:
    print(e)

# **TRAINING AND VALIDATING**

In [ ]:
# Limpando cache do Spark para treinar os modelos
spark.catalog.clearCache()
spark.sql("CLEAR CACHE")

### PREPARING DATA

In [ ]:
# Criando objeto VectorAssembler para treinar e validar os modelos
assembler = VectorAssembler(
    inputCols=[
         'DepTime',
         'CRSDepTime',
         'ArrTime',
         'CRSArrTime',
         'ActualElapsedTime',
         'CRSElapsedTime',
         'AirTime',
         'ArrDelay',
         'DepDelay',
         'Distance',
         'TaxiIn',
         'TaxiOut',
         'Month_1',
         'Month_10',
         'Month_11',
         'Month_12',
         'Month_2',
         'Month_3',
         'Month_4',
         'Month_5',
         'Month_6',
         'Month_7',
         'Month_8',
         'Month_9'
    ], outputCol='features',
)

In [ ]:
input_train = assembler.transform(train)
input_test = assembler.transform(test)

## **MULTILAYER PERCEPTRON CLASSIFIER**

### TRAINING

In [ ]:
ml = MultilayerPerceptronClassifier(
    labelCol='isDelay',
    featuresCol='features',
    layers=[24, 3, 2],
    blockSize=128,
    stepSize=0.03,
    maxIter=13,
    seed=2025
)
model = ml.fit(input_train)

In [ ]:
predictions = model.transform(input_test)

### VALIDATION

In [ ]:
for metric in ['accuracy', 'precisionByLabel', 'recallByLabel']:
    evaluator = MulticlassClassificationEvaluator(labelCol="isDelay", metricName=metric)
    print(f"{metric.upper()}:", evaluator.evaluate(predictions))

## **GAUSSIAN NAIVE BAYS**

### TRAINING

In [ ]:
nb = NaiveBayes(
    labelCol='isDelay',
    featuresCol='features',
    smoothing = 1.0,
    modelType = 'gaussian'
)
model_nb = nb.fit(input_train)

In [ ]:
predictions_nb = model_nb.transform(input_test)

### VALIDATION

In [ ]:
for metric in ['accuracy', 'precisionByLabel', 'recallByLabel']:
    evaluator = MulticlassClassificationEvaluator(labelCol="isDelay", metricName=metric)
    print(f"{metric.upper()}:", evaluator.evaluate(predictions_nb))